### 문제 정의
- 미국의 도시인 보스턴 1970년대 거주지에 대한 데이터를 바탕으로 주택 가격 예측해보기
- 답이 있는 데이터 -> 지도학습
- 연속적인 수치(가격) 예측 -> 회귀

In [1]:
# 경고창 무시 하는 코드
import warnings
warnings.filterwarnings(action='ignore')

In [3]:
# 라이브러리 불러오기
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 보스턴 주택 데이터 불러오기
# 사이킷런 모델안에 공부용으로 데이터 만들어 놓은거 불러오기
from sklearn.datasets import fetch_openml
boston = fetch_openml(name = 'boston', as_frame = True)
boston

{'data':         CRIM    ZN  INDUS CHAS    NOX     RM   AGE     DIS RAD    TAX  \
 0    0.00632  18.0   2.31    0  0.538  6.575  65.2  4.0900   1  296.0   
 1    0.02731   0.0   7.07    0  0.469  6.421  78.9  4.9671   2  242.0   
 2    0.02729   0.0   7.07    0  0.469  7.185  61.1  4.9671   2  242.0   
 3    0.03237   0.0   2.18    0  0.458  6.998  45.8  6.0622   3  222.0   
 4    0.06905   0.0   2.18    0  0.458  7.147  54.2  6.0622   3  222.0   
 ..       ...   ...    ...  ...    ...    ...   ...     ...  ..    ...   
 501  0.06263   0.0  11.93    0  0.573  6.593  69.1  2.4786   1  273.0   
 502  0.04527   0.0  11.93    0  0.573  6.120  76.7  2.2875   1  273.0   
 503  0.06076   0.0  11.93    0  0.573  6.976  91.0  2.1675   1  273.0   
 504  0.10959   0.0  11.93    0  0.573  6.794  89.3  2.3889   1  273.0   
 505  0.04741   0.0  11.93    0  0.573  6.030  80.8  2.5050   1  273.0   
 
      PTRATIO       B  LSTAT  
 0       15.3  396.90   4.98  
 1       17.8  396.90   9.14  
 2       

In [4]:
# 조회
boston.keys()

dict_keys(['data', 'target', 'frame', 'categories', 'feature_names', 'target_names', 'DESCR', 'details', 'url'])

In [5]:
print(boston.DESCR)

**Author**:   
**Source**: Unknown - Date unknown  
**Please cite**:   

The Boston house-price data of Harrison, D. and Rubinfeld, D.L. 'Hedonic
prices and the demand for clean air', J. Environ. Economics & Management,
vol.5, 81-102, 1978.   Used in Belsley, Kuh & Welsch, 'Regression diagnostics
...', Wiley, 1980.   N.B. Various transformations are used in the table on
pages 244-261 of the latter.
Variables in order:
CRIM     per capita crime rate by town
ZN       proportion of residential land zoned for lots over 25,000 sq.ft.
INDUS    proportion of non-retail business acres per town
CHAS     Charles River dummy variable (= 1 if tract bounds river; 0 otherwise)
NOX      nitric oxides concentration (parts per 10 million)
RM       average number of rooms per dwelling
AGE      proportion of owner-occupied units built prior to 1940
DIS      weighted distances to five Boston employment centres
RAD      index of accessibility to radial highways
TAX      full-value property-tax rate per $10

#### 설명변수 (원인: 예측값을 설명할 수 있는 변수)
CRIM: 범죄율  
INDUS: 비소매상업지역 면접 비율  
NOX: 일산화질소 농도  
RM : 주택당 방 수  
LSTAT: 인구 중 하위 계층 비율  
B: 인구 중 흑인 비율  
PTRATIO: 학생/교사 비율  
ZN: 25,000 평방피트를 초과 거주지역 비율  
CHAS : 찰스강의 경계에 위치한 경우는 1, 아니면 0  
AGE : 1940년 이전에 건축된 주택의 비율  
RAD : 고속도로 접근 인덱스  
DIS: 직업센터의 거리  
TAX: 재산세율  
    
#### 반응 변수(결과: 예측하고자 하는 값 )
MEDV: 주택가격 (1000달러 단위)

### 데이터 전처리
- X : 독립변수, 특성, 입력변수
- y : 종속변수, 타겟, 출력변수

In [6]:
boston.keys()

dict_keys(['data', 'target', 'frame', 'categories', 'feature_names', 'target_names', 'DESCR', 'details', 'url'])

In [7]:
# 데이터 프레임 형태로 전처리
X = pd.DataFrame(boston.data,
                 columns = boston.feature_names)
y = pd.Series(boston.target)

In [8]:
X.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33


In [9]:
y.head()
# 1000만 달러 단위

0    24.0
1    21.6
2    34.7
3    33.4
4    36.2
Name: MEDV, dtype: float64

In [10]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   CRIM     506 non-null    float64 
 1   ZN       506 non-null    float64 
 2   INDUS    506 non-null    float64 
 3   CHAS     506 non-null    category
 4   NOX      506 non-null    float64 
 5   RM       506 non-null    float64 
 6   AGE      506 non-null    float64 
 7   DIS      506 non-null    float64 
 8   RAD      506 non-null    category
 9   TAX      506 non-null    float64 
 10  PTRATIO  506 non-null    float64 
 11  B        506 non-null    float64 
 12  LSTAT    506 non-null    float64 
dtypes: category(2), float64(11)
memory usage: 44.7 KB


In [11]:
# X 특성 중에서 category 자료형을 수치형태(int)로 변환 하자
# astype() 사용
# CHAS, RAD
X[['CHAS', 'RAD']] = X[['CHAS', 'RAD']].astype('int64')


In [13]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   CRIM     506 non-null    float64
 1   ZN       506 non-null    float64
 2   INDUS    506 non-null    float64
 3   CHAS     506 non-null    int64  
 4   NOX      506 non-null    float64
 5   RM       506 non-null    float64
 6   AGE      506 non-null    float64
 7   DIS      506 non-null    float64
 8   RAD      506 non-null    int64  
 9   TAX      506 non-null    float64
 10  PTRATIO  506 non-null    float64
 11  B        506 non-null    float64
 12  LSTAT    506 non-null    float64
dtypes: float64(11), int64(2)
memory usage: 51.5 KB


### train, test 분리
- train_test_split 함수 이용
- 순서 : X_train, X_test, y_train, y_test
- 7:3 비율, random_state = 0

In [14]:
from sklearn.model_selection import train_test_split

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size = 0.3,
                                                    random_state = 0)

In [18]:
# 트레인이 잘 섞여서 나왔는지 확인
X_train

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT
141,1.62864,0.0,21.89,0,0.624,5.019,100.0,1.4394,4,437.0,21.2,396.90,34.41
272,0.11460,20.0,6.96,0,0.464,6.538,58.7,3.9175,3,223.0,18.6,394.96,7.73
135,0.55778,0.0,21.89,0,0.624,6.335,98.2,2.1107,4,437.0,21.2,394.67,16.96
298,0.06466,70.0,2.24,0,0.400,6.345,20.1,7.8278,5,358.0,14.8,368.24,4.97
122,0.09299,0.0,25.65,0,0.581,5.961,92.9,2.0869,2,188.0,19.1,378.09,17.93
...,...,...,...,...,...,...,...,...,...,...,...,...,...
323,0.28392,0.0,7.38,0,0.493,5.708,74.3,4.7211,5,287.0,19.6,391.13,11.74
192,0.08664,45.0,3.44,0,0.437,7.178,26.3,6.4798,5,398.0,15.2,390.49,2.87
117,0.15098,0.0,10.01,0,0.547,6.021,82.6,2.7474,6,432.0,17.8,394.51,10.30
47,0.22927,0.0,6.91,0,0.448,6.030,85.5,5.6894,3,233.0,17.9,392.74,18.80


In [19]:
y_train

141    14.4
272    24.4
135    18.1
298    22.5
122    20.5
       ... 
323    18.5
192    36.4
117    19.2
47     16.6
172    23.1
Name: MEDV, Length: 354, dtype: float64

### LinearRegression 사용해서 모델 학습시키기

In [21]:
# 모델 선정단계

In [20]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

In [22]:
# 모델 생성(초기화)
linear_model = LinearRegression()

In [23]:
# 학습 단계
linear_model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](13,)","[-0.12, 0.04, 0.01,...,-1.02, 0.01,-0.49]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](13,)","['CRIM','ZN','INDUS',...,'PTRATIO','B','LSTAT']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,37.94
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,13
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(13)


In [25]:
# 예측
y_pre = linear_model.predict(X_test)
y_pre

array([24.9357079 , 23.75163164, 29.32638296, 11.97534566, 21.37272478,
       19.19148525, 20.5717479 , 21.21154015, 19.04572003, 20.35463238,
        5.44119126, 16.93688709, 17.15482272,  5.3928209 , 40.20270696,
       32.31327348, 22.46213268, 36.50124666, 31.03737014, 23.17124551,
       24.74815321, 24.49939403, 20.6595791 , 30.4547583 , 22.32487164,
       10.18932894, 17.44286422, 18.26103077, 35.63299326, 20.81960303,
       18.27218007, 17.72047628, 19.33772473, 23.62254823, 28.97766856,
       19.45036239, 11.13170639, 24.81843595, 18.05294835, 15.59712226,
       26.21043403, 20.81140432, 22.17349382, 15.48367365, 22.62261604,
       24.88561528, 19.74754478, 23.0465628 ,  9.84579105, 24.36378793,
       21.47849008, 17.62118176, 24.39160873, 29.95102691, 13.57219422,
       21.53645439, 20.53306273, 15.03433182, 14.3232289 , 22.11929299,
       17.07321915, 21.54141094, 32.96766968, 31.371599  , 17.7860591 ,
       32.75069556, 18.74795323, 19.21428022, 19.41970047, 23.08

In [26]:
# 교차검증
result = cross_val_score(linear_model,
                X_train, y_train,
                cv = 5) # 5개로 나누어 교차 검증
result

array([0.7246982 , 0.58082515, 0.77515092, 0.72161474, 0.78935797])

In [27]:
result.mean()

np.float64(0.718329397431592)

### 선형 회귀 평가 지표
- MSE : 평균제곱오차
- RMSE : 평균제곱오차에 루트를 씌운
- MAE : 오차에 절댓값을 씌워 평균낸 것
- r2 : 편차에 비한 오차

In [28]:
from sklearn.metrics import mean_squared_error # 평균제곱오차
from sklearn.metrics import mean_absolute_error # 평균절대오차
from sklearn.metrics import r2_score # 결정 계수

In [29]:
# y_test <-> y_pre
# MSE 평균 제곱 오차
# mean_squared_error(실제값, 예측값)
mean_squared_error(y_test, y_pre)

27.195965766883475

In [30]:
# RMSE 평균 제곱 오차에 루트
# sklearn 모듈에 따로 RMSE도구가 없는 이유 : 루트 -> sqrt(), MSE속성 안에 squared = True/False
np.sqrt(mean_squared_error(y_test, y_pre))

np.float64(5.214975145375429)

In [31]:
# MAE 평균 절대 오차
mean_absolute_error(y_test, y_pre)
# MSE, RMSE, MAE 값만 봤을 때는 오차가 작은건지, 큰건지 파악이 불가능하다

3.609904060381831

In [33]:
# R2 score
# 결정계수
r2_score(y_test, y_pre)
# 1에 가까울수록 좋음.

0.6733825506400164

In [34]:
# 모델 자체 스코어 함수
linear_model.score(X_test, y_test)

0.6733825506400164

In [35]:
linear_model.score(X_train, y_train)

0.7645451026942549

- 현재 선형 모델은 성능이 좋지 못함.
- 모델의 성능을 개선해야할 필요가 있음!
- 선형 모델은 데이터가 많을수록 성능이 빛을 발하는 모델이다!
- 데이터를 늘려보자!

### 특성 확장
- 특성 확장을 해서 모델의 성능을 높여보자!
- 13개의 수치 데이터가 따로도 의미가 있지만, 같이 있었을 때에도 집값에 영향이 있는지 확인해보자!
- 컬럼 두개를 묶어서 하나의 특성으로 만들자는 의미.(두개의 값을 곱해서)

In [36]:
extended_X_train = X_train.copy() # 원본 데이터 안건드릴려고 복사함.

In [37]:
# 반복문을 사용하여 특성 확장
for col1 in X_train.columns : 
    for col2 in X_train.columns :
        extended_X_train[col1+'x'+col2] = X_train[col1] * X_train[col2]

In [38]:
extended_X_train

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,...,LSTATxCHAS,LSTATxNOX,LSTATxRM,LSTATxAGE,LSTATxDIS,LSTATxRAD,LSTATxTAX,LSTATxPTRATIO,LSTATxB,LSTATxLSTAT
141,1.62864,0.0,21.89,0,0.624,5.019,100.0,1.4394,4,437.0,...,0.0,21.47184,172.70379,3441.000,49.529754,137.64,15037.17,729.492,13657.3290,1184.0481
272,0.11460,20.0,6.96,0,0.464,6.538,58.7,3.9175,3,223.0,...,0.0,3.58672,50.53874,453.751,30.282275,23.19,1723.79,143.778,3053.0408,59.7529
135,0.55778,0.0,21.89,0,0.624,6.335,98.2,2.1107,4,437.0,...,0.0,10.58304,107.44160,1665.472,35.797472,67.84,7411.52,359.552,6693.6032,287.6416
298,0.06466,70.0,2.24,0,0.400,6.345,20.1,7.8278,5,358.0,...,0.0,1.98800,31.53465,99.897,38.904166,24.85,1779.26,73.556,1830.1528,24.7009
122,0.09299,0.0,25.65,0,0.581,5.961,92.9,2.0869,2,188.0,...,0.0,10.41733,106.88073,1665.697,37.418117,35.86,3370.84,342.463,6779.1537,321.4849
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
323,0.28392,0.0,7.38,0,0.493,5.708,74.3,4.7211,5,287.0,...,0.0,5.78782,67.01192,872.282,55.425714,58.70,3369.38,230.104,4591.8662,137.8276
192,0.08664,45.0,3.44,0,0.437,7.178,26.3,6.4798,5,398.0,...,0.0,1.25419,20.60086,75.481,18.597026,14.35,1142.26,43.624,1120.7063,8.2369
117,0.15098,0.0,10.01,0,0.547,6.021,82.6,2.7474,6,432.0,...,0.0,5.63410,62.01630,850.780,28.298220,61.80,4449.60,183.340,4063.4530,106.0900
47,0.22927,0.0,6.91,0,0.448,6.030,85.5,5.6894,3,233.0,...,0.0,8.42240,113.36400,1607.400,106.960720,56.40,4380.40,336.520,7383.5120,353.4400


In [39]:
# X_test 특성 확장
extended_X_test = X_test.copy()

In [40]:
for col1 in X_test.columns : 
    for col2 in X_test.columns :
        extended_X_test[col1+'x'+col2] = X_test[col1] * X_test[col2]

In [42]:
extended_X_test

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,...,LSTATxCHAS,LSTATxNOX,LSTATxRM,LSTATxAGE,LSTATxDIS,LSTATxRAD,LSTATxTAX,LSTATxPTRATIO,LSTATxB,LSTATxLSTAT
329,0.06724,0.0,3.24,0,0.460,6.333,17.2,5.2146,4,430.0,...,0.0,3.37640,46.48422,126.248,38.275164,29.36,3156.20,124.046,2754.0414,53.8756
371,9.23230,0.0,18.10,0,0.631,6.216,100.0,1.1691,24,666.0,...,0.0,6.01343,59.23848,953.000,11.141523,228.72,6346.98,192.506,3489.4095,90.8209
219,0.11425,0.0,13.89,1,0.550,6.373,92.4,3.3633,5,276.0,...,10.5,5.77500,66.91650,970.200,35.314650,52.50,2898.00,172.200,4134.2700,110.2500
403,24.80170,0.0,18.10,0,0.693,5.349,96.0,1.7028,24,666.0,...,0.0,13.70061,105.74973,1897.920,33.664356,474.48,13166.82,399.354,7846.7130,390.8529
78,0.05646,0.0,12.83,0,0.437,6.232,53.7,5.0141,5,398.0,...,0.0,5.39258,76.90288,662.658,61.873994,61.70,4911.32,230.758,4768.1760,152.2756
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,...,0.0,2.44114,38.09351,288.886,32.311526,15.99,1183.26,99.671,2115.4770,28.4089
428,7.36711,0.0,18.10,0,0.679,6.193,78.1,1.9356,24,666.0,...,0.0,14.61208,133.27336,1680.712,41.654112,516.48,14332.32,434.704,2081.6296,463.1104
385,16.81180,0.0,18.10,0,0.700,5.277,98.1,1.4261,24,666.0,...,0.0,21.56700,162.58437,3022.461,43.938141,739.44,20519.46,622.362,12228.4890,949.2561
308,0.49298,0.0,9.90,0,0.544,6.635,82.5,3.3175,4,304.0,...,0.0,2.46976,30.12290,374.550,15.061450,18.16,1380.16,83.536,1801.9260,20.6116


In [43]:
extended_X_test.info()

<class 'pandas.DataFrame'>
Index: 152 entries, 329 to 5
Columns: 182 entries, CRIM to LSTATxLSTAT
dtypes: float64(176), int64(6)
memory usage: 217.3 KB


In [44]:
extended_X_test.columns

Index(['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX',
       ...
       'LSTATxCHAS', 'LSTATxNOX', 'LSTATxRM', 'LSTATxAGE', 'LSTATxDIS',
       'LSTATxRAD', 'LSTATxTAX', 'LSTATxPTRATIO', 'LSTATxB', 'LSTATxLSTAT'],
      dtype='str', length=182)

In [45]:
# 확장된 데이터로 다시 모델을 학습시켜 보자!!
linear_model2 = LinearRegression()
linear_model2.fit(extended_X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](182,)","[ 0.02,-0.04, 0.61,..., 0.01,-0. , 0.03]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](182,)","['CRIM','ZN','INDUS',...,'LSTATxPTRATIO','LSTATxB','LSTATxLSTAT']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,26.44
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,182
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(86)


In [46]:
linear_model2.score(extended_X_test, y_test)

0.6681396381286437

In [ ]:
# 특성이 너어어어무 많아져서 과대적합이 와버렸구나...........
# 정규화가 정답이다!!

#### 정규화
Lasso: L1 규제를 사용하여 중요도가 낮은 특성의 가중치를 아예 0으로 만들어 불필요한 특성을 제거하고, 모델을 단순하게 만들어 줍니다.

Ridge: L2 규제를 사용하여 모든 특성의 가중치 크기를 전반적으로 작게 줄여주고, 특정 특성이 과도한 영향을 미치지 못하도록 억제하여 과대적합을 막아줍니다.

In [47]:
# L1 규제 : Lasso
from sklearn.linear_model import Lasso

In [52]:
# 모델 생성(초기화)
lasso_model = Lasso(alpha = 10) # alpha : 얼마나 규제를 가할건지 -> 1.0 기본값

# 규제가 강해지면 과대적합을 줄일 수 있다. 하지만 오차가 커질 가능성이 있다. 

In [53]:
# 학습
lasso_model.fit(extended_X_train, y_train)

,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",10
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary <warm_start>`.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'
Name,Type,Value


In [54]:
lasso_model.score(extended_X_test, y_test)

0.7467008426432002

In [55]:
# L2 규제 : Ridge
from sklearn.linear_model import Ridge

In [56]:
# 모델 생성(초기화)
ridge_model = Ridge(alpha = 100)

In [57]:
# 학습
ridge_model.fit(extended_X_train, y_train)

,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",100
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' 

In [58]:
ridge_model.score(extended_X_test, y_test)

0.7767755128062253

In [ ]:
# 규제를 적용한 결과 점수가 더 올랐다!
# Lasso 처럼 변수가 소거 되는 것보다, ridge 처럼 모든 변수의 가중치를 부드럽게 줄여나가는 방식이
# 모델의 안정성과 성능 면에서 더 적합했기 때문에
# 오늘은 릿지 규제를 쓰는게 적당했다! 좋았다

In [ ]:
# Lasso를 쓰는 경우?
# 1. 중요하지 않은 특성(노이즈)을 제거하고 싶을 때
# 2. 강한 독립변수들(특성들)만 남기고 싶을 때